# Uniform Cost Search: Self-Driving Trailer, Nairobi to Kisumu
**Unit:** Artificial Intelligence | **Practical Exercise 3** | **Part 1 of 3: Formulation, Graph and Algorithm**

This notebook continues from Part 1 of the practical (PEAS, directed graph and path costs). Together, the three notebooks run a **Uniform Cost Search (UCS)** from Nairobi to Kisumu and walk through the traversal the same way it was done in class: a frontier of paths with their costs `g(n)`, expanding the lowest-cost path each time, and **only stopping when a goal is chosen for expansion**, not when it is first generated.

**Contents of the three notebooks**
1. Problem formulation 
2. The graph and path costs 
3. The UCS algorithm 
4. Step-by-step traversal 
5. Optimal path and cost 
6. Verification and visualisation 
7. Evidence of group work 

## 1. Problem formulation

| Element | Definition |
|---|---|
| **Initial state** | Trailer is in Nairobi |
| **Goal state** | Trailer is in Kisumu |
| **States** | A town on the highway network |
| **Actions** | Drive along a highway to a directly connected next town |
| **Transition model** | `RESULT(town, drive to X) = X` |
| **Step cost** | Driving distance (km) between the two towns, rounded to the nearest km (Google Maps) |
| **Path cost** | Sum of step costs along the path, `g(n)` |

UCS expands the node with the **lowest path cost g(n)**, using a **priority queue** ordered by `g`. Because every step cost is positive, the first time Kisumu is *selected for expansion*, its path is the optimal one.

## 2. Directed graph and path costs (from Part 1)
The 21 edges below are the same as in the Part 1 table (Question 3, Part A).

In [1]:
# (from, to, distance in km) -- the 21 directed edges from Part 1
EDGES = [
    ("Nairobi",    "Ruiru",      25),
    ("Nairobi",    "Athi River", 30),
    ("Nairobi",    "Naivasha",   91),
    ("Nairobi",    "Narok",     143),
    ("Ruiru",      "Thika",      21),
    ("Thika",      "Karatina",   88),
    ("Karatina",   "Nyeri",      25),
    ("Nyeri",      "Nyahururu", 104),
    ("Nyahururu",  "Nakuru",     68),
    ("Athi River", "Makindu",   149),
    ("Athi River", "Kajiado",    50),
    ("Naivasha",   "Gilgil",     30),
    ("Gilgil",     "Nakuru",     42),
    ("Nakuru",     "Kericho",   107),
    ("Kericho",    "Kisumu",     84),
    ("Narok",      "Bomet",      81),
    ("Bomet",      "Kisii",      82),
    ("Kisii",      "Kisumu",    114),
    ("Kisii",      "Migori",     67),
    ("Kisii",      "Homa Bay",   52),
    ("Homa Bay",   "Kisumu",    108),
]

# Adjacency list: town -> list of (next town, km), in the same order as the table
graph = {}
for a, b, km in EDGES:
    graph.setdefault(a, []).append((b, km))
    graph.setdefault(b, [])          # make sure dead-end towns exist as keys

START, GOAL = "Nairobi", "Kisumu"

print(f"{len(graph)} towns, {len(EDGES)} edges")
for town, nbrs in graph.items():
    print(f"{town:11s} -> {nbrs if nbrs else 'dead end'}")

19 towns, 21 edges
Nairobi     -> [('Ruiru', 25), ('Athi River', 30), ('Naivasha', 91), ('Narok', 143)]
Ruiru       -> [('Thika', 21)]
Athi River  -> [('Makindu', 149), ('Kajiado', 50)]
Naivasha    -> [('Gilgil', 30)]
Narok       -> [('Bomet', 81)]
Thika       -> [('Karatina', 88)]
Karatina    -> [('Nyeri', 25)]
Nyeri       -> [('Nyahururu', 104)]
Nyahururu   -> [('Nakuru', 68)]
Nakuru      -> [('Kericho', 107)]
Makindu     -> dead end
Kajiado     -> dead end
Gilgil      -> [('Nakuru', 42)]
Kericho     -> [('Kisumu', 84)]
Kisumu      -> dead end
Bomet       -> [('Kisii', 82)]
Kisii       -> [('Kisumu', 114), ('Migori', 67), ('Homa Bay', 52)]
Migori      -> dead end
Homa Bay    -> [('Kisumu', 108)]


## 3. The UCS algorithm

```
frontier  <- priority queue ordered by g, containing the start path (g = 0)
explored  <- empty set
loop:
    if frontier is empty: FAIL
    pop the path with the LOWEST g            <- tie: the one added first
    if its last town is the GOAL: return it   <- goal TEST on expansion
    if its last town is already explored: discard it
    add the town to explored
    for each connected town not yet explored:
        push (g + step cost, path + town) onto the frontier
```

**Why not stop when the goal is first generated?** In class, `1-9-10` reached the goal with `g = 10`, but `1-2-3-4-10` reached it with `g = 9`. A goal can be generated by an expensive path before a cheaper one is found, so UCS only stops when the goal has the lowest `g` on the frontier.

Ties are broken by insertion order (the first path added to the frontier is expanded first), matching the rule used in class.

In [1]:
import heapq
from itertools import count


def uniform_cost_search(graph, start, goal, verbose=True):
    """
    Uniform Cost Search on a weighted directed graph.
    Returns (path, cost, stats), or (None, inf, stats) if the goal is unreachable.
    """
    tie = count()                                   # insertion order for tie-breaking
    frontier = [(0, next(tie), start, [start])]     # (g, tie, town, path)
    explored = set()
    stats = {"expanded": 0, "generated": 1}
    step = 0

    def show_frontier():
        items = sorted(frontier)                    # sorted by g, then insertion order
        for g, _, _, path in items:
            print(f"      {' -> '.join(path)}   g = {g}")

    while frontier:
        step += 1
        if verbose:
            print(f"\n--- Step {step} ---")
            print("   Frontier (lowest g first):")
            show_frontier()

        g, _, town, path = heapq.heappop(frontier)  # expand the LOWEST path cost

        if town in explored:
            if verbose:
                print(f"   Discard {' -> '.join(path)} (g = {g}): {town} was already expanded at a lower or equal cost")
            continue

        if verbose:
            print(f"   >> Expand {' -> '.join(path)}  (g = {g})")

        # Goal test happens when a node is CHOSEN for expansion
        if town == goal:
            if verbose:
                print(f"   *** GOAL REACHED with the lowest g on the frontier: g = {g} ***")
            return path, g, stats

        explored.add(town)
        stats["expanded"] += 1

        if not graph[town]:
            if verbose:
                print(f"   {town} is a dead end: nothing to add to the frontier")
            continue

        for nxt, km in graph[town]:
            if nxt in explored:
                continue
            new_g = g + km
            heapq.heappush(frontier, (new_g, next(tie), nxt, path + [nxt]))
            stats["generated"] += 1
            if verbose:
                note = ""
                if nxt == goal:
                    note = "   <- goal generated, but we do NOT stop: a cheaper path may exist"
                print(f"      generate {' -> '.join(path + [nxt])}: g = {g} + {km} = {new_g}{note}")

    return None, float("inf"), stats

# Uniform Cost Search: Self-Driving Trailer, Nairobi to Kisumu
**Unit:** Artificial Intelligence (BBT 3201) | **Practical Exercise 3** | **Part 2 of 3: Traversal and Result**

Continues from Part 1. This part runs the UCS function and records the step-by-step traversal and the optimal path.

**Contents of this notebook**
4. Step-by-step traversal
5. Optimal path and cost

**Needs from Part 1:** the `graph`, `EDGES`, `START` and `GOAL` variables (Section 2) and the `uniform_cost_search` function (Section 3). Run Part 1's code cells first, or paste them above these cells.

## 4. Step-by-step traversal
Each step prints the current frontier, the path chosen for expansion, and the new paths generated, the same way the class example was traced.

In [3]:
path, cost, stats = uniform_cost_search(graph, START, GOAL, verbose=True)


--- Step 1 ---
   Frontier (lowest g first):
      Nairobi   g = 0
   >> Expand Nairobi  (g = 0)
      generate Nairobi -> Ruiru: g = 0 + 25 = 25
      generate Nairobi -> Athi River: g = 0 + 30 = 30
      generate Nairobi -> Naivasha: g = 0 + 91 = 91
      generate Nairobi -> Narok: g = 0 + 143 = 143

--- Step 2 ---
   Frontier (lowest g first):
      Nairobi -> Ruiru   g = 25
      Nairobi -> Athi River   g = 30
      Nairobi -> Naivasha   g = 91
      Nairobi -> Narok   g = 143
   >> Expand Nairobi -> Ruiru  (g = 25)
      generate Nairobi -> Ruiru -> Thika: g = 25 + 21 = 46

--- Step 3 ---
   Frontier (lowest g first):
      Nairobi -> Athi River   g = 30
      Nairobi -> Ruiru -> Thika   g = 46
      Nairobi -> Naivasha   g = 91
      Nairobi -> Narok   g = 143
   >> Expand Nairobi -> Athi River  (g = 30)
      generate Nairobi -> Athi River -> Makindu: g = 30 + 149 = 179
      generate Nairobi -> Athi River -> Kajiado: g = 30 + 50 = 80

--- Step 4 ---
   Frontier (lowest g first

## 5. Optimal path and cost

In [4]:
print("=" * 70)
print("UNIFORM COST SEARCH RESULT")
print("=" * 70)
print(f"Start : {START}")
print(f"Goal  : {GOAL}")
print(f"Optimal path : {' -> '.join(path)}")
print(f"Total cost   : {cost} km")
print()

# Break the cost down edge by edge
lookup = {(a, b): km for a, b, km in EDGES}
running = 0
for a, b in zip(path, path[1:]):
    running += lookup[(a, b)]
    print(f"  {a:10s} -> {b:10s} {lookup[(a, b)]:4d} km   (cumulative {running} km)")
print(f"\nCalculation: {' + '.join(str(lookup[(a, b)]) for a, b in zip(path, path[1:]))} = {cost} km")
print(f"Towns expanded: {stats['expanded']}, paths generated: {stats['generated']}")

UNIFORM COST SEARCH RESULT
Start : Nairobi
Goal  : Kisumu
Optimal path : Nairobi -> Naivasha -> Gilgil -> Nakuru -> Kericho -> Kisumu
Total cost   : 354 km

  Nairobi    -> Naivasha     91 km   (cumulative 91 km)
  Naivasha   -> Gilgil       30 km   (cumulative 121 km)
  Gilgil     -> Nakuru       42 km   (cumulative 163 km)
  Nakuru     -> Kericho     107 km   (cumulative 270 km)
  Kericho    -> Kisumu       84 km   (cumulative 354 km)

Calculation: 91 + 30 + 42 + 107 + 84 = 354 km
Towns expanded: 16, paths generated: 20
